In [2]:
# =============================================================================
# WildChat Analytics Platform — Python ETL Pipeline
# Group G-6 (Section A) | Rishihood University | Data Analytics Capstone 2025
# Google Colab — Run cells top to bottom
# =============================================================================

# ===========================================================================
# CELL 1 — Install Dependencies
# ===========================================================================
# Run this cell first. Runtime will restart after install — that is expected.

!pip install -q datasets transformers vaderSentiment langdetect nltk scikit-learn \
             pandas numpy tqdm openpyxl accelerate

# ===========================================================================
# CELL 2 — Imports & NLTK Downloads
# ===========================================================================

import os
import re
import json
import warnings
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from datetime import datetime

# NLP
import nltk
from nltk.tokenize import word_tokenize
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from langdetect import detect, LangDetectException

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# HuggingFace
from datasets import load_dataset
from transformers import pipeline

warnings.filterwarnings("ignore")

# Download NLTK data
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

print("✅ All imports successful.")

✅ All imports successful.


In [3]:
# ===========================================================================
# CELL 3 — Configuration
# ===========================================================================

CFG = {
    "dataset_name"   : "allenai/WildChat-1M",
    "split"          : "train",
    "batch_size"     : 50_000,
    "max_batches"    : 20,          # 20 × 50k = 1 M rows  (reduce for testing)
    "sample_batches" : 2,           # ← SET TO 2 FOR QUICK TEST (~100k rows)
    "output_dir"     : "/content/wildchat_outputs",
    "min_content_len": 3,           # drop bot probes shorter than this
    "power_user_pct" : 0.90,        # top 10% = power users
    "kmeans_k"       : 5,
    "tfidf_max_feat" : 5_000,
    "prompt_categories": [
        "coding and technical",
        "factual question answering",
        "creative writing",
        "emotional support",
        "roleplay or persona",
        "harmful or policy violating",
        "other",
    ],
}

# ⚠️  For a FULL 1M run: set sample_batches = max_batches = 20
# ⚠️  For a QUICK TEST:  set sample_batches = 2  (≈100k rows, ~5 min on Colab)

os.makedirs(CFG["output_dir"], exist_ok=True)
print(f"✅ Output directory: {CFG['output_dir']}")

✅ Output directory: /content/wildchat_outputs


In [10]:
# ===========================================================================
# CELL 4 — STAGE 1: EXTRACTION
# ===========================================================================

print("\n" + "="*60)
print("STAGE 1 — EXTRACTION")
print("="*60)

print("⏳ Loading dataset in streaming mode from HuggingFace Hub...")
ds = load_dataset(CFG["dataset_name"], split=CFG["split"], streaming=True)

batches = []
n_batches = CFG["sample_batches"]   # change to max_batches for full run

with tqdm(total=n_batches, desc="Loading batches") as pbar:
    for i, batch in enumerate(ds.iter(batch_size=CFG["batch_size"])):
        df_batch = pd.DataFrame(batch)
        batches.append(df_batch)
        pbar.update(1)
        if i + 1 >= n_batches:
            break

df_raw = pd.concat(batches, ignore_index=True)
print(f"\n✅ Extracted {len(df_raw):,} rows | Columns: {list(df_raw.columns)}")
print(df_raw.dtypes)


STAGE 1 — EXTRACTION
⏳ Loading dataset in streaming mode from HuggingFace Hub...


Loading batches:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Extracted 100,000 rows | Columns: ['conversation_hash', 'model', 'timestamp', 'conversation', 'turn', 'language', 'openai_moderation', 'detoxify_moderation', 'toxic', 'redacted', 'state', 'country', 'hashed_ip', 'header']
conversation_hash                   object
model                               object
timestamp              datetime64[ns, UTC]
conversation                        object
turn                                 int64
language                            object
openai_moderation                   object
detoxify_moderation                 object
toxic                                 bool
redacted                              bool
state                               object
country                             object
hashed_ip                           object
header                              object
dtype: object


In [19]:
# ===========================================================================
# CELL 5 — STAGE 2: CLEANING
# ===========================================================================

print("\n" + "="*60)
print("STAGE 2 — CLEANING")
print("="*60)

df = df_raw.copy()
initial_rows = len(df)

# ── 2.1  Flatten conversation list-of-dicts into one row per turn ──────────
# WildChat stores conversation as a list of {role, content, ...} dicts.
# We explode it so each turn becomes its own row.

if "conversation" in df.columns:
    print("  Exploding conversation column...")
    # Assign conversation_id from conversation_hash before exploding
    df["conversation_id"] = df["conversation_hash"]

    # Explode the conversation column. The index will be repeated for each exploded row.
    df = df.explode("conversation")

    # Now, generate turn_id using cumcount() on the original (repeated) index
    # We use `groupby(level=0)` to group by the original index, which effectively groups by conversation.
    df["turn_id"] = df.groupby(level=0).cumcount()

    # Extract role and content from the conversation dictionary
    conv_data = df["conversation"].apply(
        lambda x: x if isinstance(x, dict) else {}
    )
    df["role"]    = conv_data.apply(lambda x: x.get("role", None))
    df["content"] = conv_data.apply(lambda x: x.get("content", ""))
    df.drop(columns=["conversation"], inplace=True) # Drop the original conversation column
    df.reset_index(drop=True, inplace=True) # Reset the index after all manipulations

# ── 2.2  Keep only expected columns (graceful handling) ────────────────────
EXPECTED = [
    "conversation_id", "turn_id", "timestamp", "role", "content",
    "country", "state", "model", "toxic", "redacted", "language",
    "openai_moderation",
]
for col in EXPECTED:
    if col not in df.columns:
        df[col] = None   # create missing columns as None

# ── 2.3  Deduplication ──────────────────────────────────────────────────────
df.drop_duplicates(subset=["conversation_id", "turn_id"], inplace=True)

# ── 2.4  Fill missing values ────────────────────────────────────────────────
df["country"]  = df["country"].fillna("UNKNOWN")
df["language"] = df["language"].fillna("und")
df["state"]    = df["state"].fillna("UNKNOWN")
df["model"]    = df["model"].fillna("unknown")

# ── 2.5  Parse timestamps ────────────────────────────────────────────────────
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

# ── 2.6  Cast toxic to int ───────────────────────────────────────────────────
df["toxic"] = pd.to_numeric(df["toxic"], errors="coerce").fillna(0).astype(int)

# ── 2.7  Drop rows with content shorter than min_content_len ────────────────
df["content"] = df["content"].fillna("").astype(str)
df = df[df["content"].str.len() >= CFG["min_content_len"]]

# ── 2.8  Drop rows missing conversation_id or timestamp ─────────────────────
df = df.dropna(subset=["conversation_id"])

# ── 2.9  Normalise model names ───────────────────────────────────────────────
df["model"] = df["model"].str.replace(r"gpt-3\.5-turbo.*", "gpt-3.5-turbo",
                                       regex=True)
df["model"] = df["model"].str.replace(r"gpt-4.*", "gpt-4", regex=True)

print(f"  Rows after cleaning: {len(df):,}  (removed {initial_rows - len(df):,})")
print(f"  Unique conversations: {df['conversation_id'].nunique():,}")
print("✅ Stage 2 complete.")


STAGE 2 — CLEANING
  Exploding conversation column...
  Rows after cleaning: 556,384  (removed -456,384)
  Unique conversations: 99,110
✅ Stage 2 complete.


In [20]:
# ===========================================================================
# CELL 6 — STAGE 3: TEXT PREPROCESSING
# ===========================================================================

print("\n" + "="*60)
print("STAGE 3 — TEXT PREPROCESSING")
print("="*60)

# ── 3.1  Basic text normalisation ────────────────────────────────────────────
def clean_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    # Remove PII redaction placeholders
    text = re.sub(r"\[NAME\]|\[EMAIL\]|\[PHONE\]|\[ADDRESS\]|\[DATE\]",
                  "", text, flags=re.IGNORECASE)
    return text.strip()

print("  Cleaning text content...")
df["content_clean"] = df["content"].apply(clean_text)

# ── 3.2  Language detection (fill missing 'und' values) ─────────────────────
def safe_detect(text: str) -> str:
    try:
        return detect(text[:500])   # limit to 500 chars for speed
    except LangDetectException:
        return "und"

und_mask = df["language"] == "und"
print(f"  Detecting language for {und_mask.sum():,} rows with missing language...")
df.loc[und_mask, "language"] = df.loc[und_mask, "content_clean"].apply(safe_detect)

# ── 3.3  Token count (using whitespace split for speed; NLTK for accuracy) ──
print("  Counting tokens...")
df["token_count"] = df["content_clean"].apply(lambda x: len(x.split()))

# ── 3.4  VADER Sentiment on user messages only ───────────────────────────────
print("  Running VADER sentiment analysis on user messages...")
vader = SentimentIntensityAnalyzer()

def get_sentiment(text: str) -> float:
    return vader.polarity_scores(text)["compound"]

user_mask = df["role"] == "user"
df.loc[user_mask, "sentiment_score"] = (
    df.loc[user_mask, "content_clean"].apply(get_sentiment)
)
df["sentiment_score"] = df["sentiment_score"].fillna(0.0)

print("✅ Stage 3 complete.")


STAGE 3 — TEXT PREPROCESSING
  Cleaning text content...
  Detecting language for 0 rows with missing language...
  Counting tokens...
  Running VADER sentiment analysis on user messages...
✅ Stage 3 complete.


In [21]:
# ===========================================================================
# CELL 7 — STAGE 4: FEATURE ENGINEERING
# ===========================================================================

print("\n" + "="*60)
print("STAGE 4 — FEATURE ENGINEERING")
print("="*60)

# ── 4.1  Parse openai_moderation JSON ────────────────────────────────────────
def parse_moderation(mod_val) -> float:
    """Aggregate moderation category scores into a composite toxicity score."""
    if mod_val is None or (isinstance(mod_val, float) and np.isnan(mod_val)):
        return 0.0
    try:
        if isinstance(mod_val, str):
            mod_val = json.loads(mod_val)
        if isinstance(mod_val, list) and len(mod_val) > 0:
            mod_val = mod_val[0]
        if isinstance(mod_val, dict):
            cats = mod_val.get("category_scores", {})
            if cats:
                return float(np.mean(list(cats.values())))
    except Exception:
        pass
    return 0.0

print("  Parsing openai_moderation scores...")
df["toxicity_score"] = df["openai_moderation"].apply(parse_moderation)

# ── 4.2  Time-based features ─────────────────────────────────────────────────
print("  Extracting time features...")
df["hour_of_day"]  = df["timestamp"].dt.hour.fillna(-1).astype(int)
df["day_of_week"]  = df["timestamp"].dt.dayofweek.fillna(-1).astype(int)
df["week_num"]     = df["timestamp"].dt.isocalendar().week.astype(int)
df["date"]         = df["timestamp"].dt.date

# ── 4.3  Conversation-level aggregates ──────────────────────────────────────
print("  Computing conversation-level features...")

conv_agg = (
    df.groupby("conversation_id")
    .agg(
        conv_turn_count   = ("turn_id",        "nunique"),
        user_msg_count    = ("role",            lambda x: (x == "user").sum()),
        avg_prompt_len    = ("token_count",     lambda x:
                              x[df.loc[x.index, "role"] == "user"].mean()
                              if (df.loc[x.index, "role"] == "user").any() else 0),
        avg_response_len  = ("token_count",     lambda x:
                              x[df.loc[x.index, "role"] == "assistant"].mean()
                              if (df.loc[x.index, "role"] == "assistant").any() else 0),
        sentiment_mean    = ("sentiment_score", "mean"),
        toxicity_max      = ("toxicity_score",  "max"),
        toxic_flag        = ("toxic",           "max"),
        ts_min            = ("timestamp",       "min"),
        ts_max            = ("timestamp",       "max"),
        country           = ("country",         "first"),
        language          = ("language",        "first"),
        model             = ("model",           "first"),
        hour_of_day       = ("hour_of_day",     "first"),
        day_of_week       = ("day_of_week",     "first"),
        week_num          = ("week_num",        "first"),
    )
    .reset_index()
)

# Session duration
conv_agg["session_duration_min"] = (
    (conv_agg["ts_max"] - conv_agg["ts_min"])
    .dt.total_seconds()
    .div(60)
    .fillna(0)
    .clip(lower=0)
)

# ── 4.4  Drop-off flag ────────────────────────────────────────────────────────
conv_agg["drop_off_flag"] = (conv_agg["conv_turn_count"] <= 2).astype(int)

# ── 4.5  Response Quality Score (heuristic 0–10) ─────────────────────────────
def quality_score(row) -> float:
    """
    Weighted heuristic:
      30% — response length appropriateness (penalise very short / very long)
      40% — lexical diversity proxy (avg_response_len as stand-in; capped)
      30% — coherence proxy (longer conversations score higher)
    """
    # Length score: ideal 50–300 tokens → 10; falls off outside range
    rl = row["avg_response_len"]
    if rl < 10:
        len_score = 1.0
    elif rl <= 300:
        len_score = min(10.0, rl / 30.0)
    else:
        len_score = max(1.0, 10.0 - (rl - 300) / 200.0)

    # Diversity score: longer avg response = more diverse (proxy)
    div_score = min(10.0, rl / 50.0)

    # Coherence: more turns → higher coherence
    coh_score = min(10.0, row["conv_turn_count"] * 1.5)

    return round(0.30 * len_score + 0.40 * div_score + 0.30 * coh_score, 2)

conv_agg["response_quality_score"] = conv_agg.apply(quality_score, axis=1)

# ── 4.6  Power-user flag (by user = conversation_id proxy here) ──────────────
threshold = conv_agg["conv_turn_count"].quantile(CFG["power_user_pct"])
conv_agg["is_power_user"] = (conv_agg["conv_turn_count"] > threshold).astype(int)

print(f"  Power user threshold (turn count > {threshold:.0f}): "
      f"{conv_agg['is_power_user'].sum():,} conversations")

print("✅ Stage 4 complete.")



STAGE 4 — FEATURE ENGINEERING
  Parsing openai_moderation scores...
  Extracting time features...
  Computing conversation-level features...
  Power user threshold (turn count > 14): 7,880 conversations
✅ Stage 4 complete.


In [22]:
# ===========================================================================
# CELL 8 — ZERO-SHOT PROMPT CLASSIFICATION  (sample for speed)
# ===========================================================================

print("\n" + "="*60)
print("ZERO-SHOT PROMPT CLASSIFICATION")
print("="*60)

# For speed we classify a stratified 10% sample of user prompts
user_prompts = df[df["role"] == "user"][["conversation_id", "content_clean"]].copy()

SAMPLE_SIZE = min(10_000, len(user_prompts))   # cap at 10k for Colab GPU/CPU
sample_prompts = user_prompts.sample(SAMPLE_SIZE, random_state=42)

print(f"  Classifying {SAMPLE_SIZE:,} prompts (zero-shot)...")
try:
    classifier = pipeline(
        "zero-shot-classification",
        model="typeform/distilbart-mnli-12-3",
        device=-1,         # CPU; set device=0 for GPU
    )

    def classify_prompt(text: str) -> str:
        text = text[:512]  # truncate to model max
        result = classifier(text, CFG["prompt_categories"], multi_label=False)
        return result["labels"][0]

    tqdm.pandas(desc="Classifying")
    sample_prompts["prompt_category"] = sample_prompts["content_clean"].progress_apply(
        classify_prompt
    )
except Exception as e:
    print(f"  ⚠️  Zero-shot classifier unavailable ({e}). Using keyword fallback.")
    KEYWORDS = {
        "coding and technical"        : r"\b(code|python|error|function|debug|sql|script|api)\b",
        "factual question answering"  : r"\b(what|who|when|where|how|explain|define|list)\b",
        "creative writing"            : r"\b(write|story|poem|essay|create|fiction|imagine)\b",
        "emotional support"           : r"\b(feel|sad|anxious|depressed|lonely|help me|stress)\b",
        "roleplay or persona"         : r"\b(pretend|roleplay|act as|you are|character)\b",
        "harmful or policy violating" : r"\b(hack|illegal|drug|weapon|kill|harm|bypass)\b",
    }
    def keyword_classify(text: str) -> str:
        for cat, pattern in KEYWORDS.items():
            if re.search(pattern, text, re.IGNORECASE):
                return cat
        return "other"
    sample_prompts["prompt_category"] = sample_prompts["content_clean"].apply(
        keyword_classify
    )

# Merge category back to conv_agg
cat_map = sample_prompts.groupby("conversation_id")["prompt_category"].first()
conv_agg["prompt_category"] = conv_agg["conversation_id"].map(cat_map).fillna("other")

print("  Category distribution:")
print(conv_agg["prompt_category"].value_counts(normalize=True).mul(100).round(1))
print("✅ Classification complete.")


ZERO-SHOT PROMPT CLASSIFICATION
  Classifying 10,000 prompts (zero-shot)...
  ⚠️  Zero-shot classifier unavailable (typeform/distilbart-mnli-12-3 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`). Using keyword fallback.
  Category distribution:
prompt_category
other                          97.2
factual question answering      1.5
coding and technical            0.7
creative writing                0.4
roleplay or persona             0.1
emotional support               0.0
harmful or policy violating     0.0
Name: proportion, dtype: float64
✅ Classification complete.


In [23]:
# ===========================================================================
# CELL 9 — K-MEANS USER SEGMENTATION
# ===========================================================================

print("\n" + "="*60)
print("K-MEANS USER SEGMENTATION")
print("="*60)

SEG_FEATURES = [
    "conv_turn_count", "avg_prompt_len", "avg_response_len",
    "toxicity_max", "sentiment_mean", "session_duration_min",
    "response_quality_score",
]

seg_df = conv_agg[SEG_FEATURES].copy().fillna(0)

# Standardise
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(seg_df)

# K-Means
km = KMeans(n_clusters=CFG["kmeans_k"], random_state=42, n_init=10)
conv_agg["cluster_id"] = km.fit_predict(X_scaled)

# PCA 2D projection for Tableau scatter plot
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
conv_agg["pca_x"] = pca_coords[:, 0]
conv_agg["pca_y"] = pca_coords[:, 1]

print("  Cluster sizes:")
print(conv_agg["cluster_id"].value_counts().sort_index())

CLUSTER_LABELS = {
    0: "Power Users",
    1: "Task-Oriented",
    2: "Casual Explorers",
    3: "At-Risk Users",
    4: "Adversarial / Abuse",
}
conv_agg["cluster_label"] = conv_agg["cluster_id"].map(CLUSTER_LABELS)
print("✅ Clustering complete.")


K-MEANS USER SEGMENTATION
  Cluster sizes:
cluster_id
0     1606
1    52438
2    30591
3    12971
4     1504
Name: count, dtype: int64
✅ Clustering complete.


In [24]:
# ===========================================================================
# CELL 10 — ANOMALY DETECTION ON DAILY KPIs
# ===========================================================================

print("\n" + "="*60)
print("ANOMALY DETECTION — DAILY KPIs")
print("="*60)

daily = (
    conv_agg.groupby(
        conv_agg["ts_min"].dt.date.rename("date")
    )
    .agg(
        total_conversations = ("conversation_id", "count"),
        engagement_rate     = ("drop_off_flag",
                                lambda x: round((1 - x.mean()) * 100, 2)),
        drop_off_rate       = ("drop_off_flag",
                                lambda x: round(x.mean() * 100, 2)),
        toxicity_rate       = ("toxic_flag",
                                lambda x: round(x.mean() * 100, 2)),
        avg_turn_count      = ("conv_turn_count", "mean"),
        avg_quality_score   = ("response_quality_score", "mean"),
    )
    .reset_index()
    .sort_values("date")
)

# Isolation Forest anomaly detection
if len(daily) >= 10:
    iso = IsolationForest(contamination=0.05, random_state=42)
    kpi_cols = ["total_conversations", "drop_off_rate", "toxicity_rate"]
    daily["anomaly_flag"]  = (iso.fit_predict(daily[kpi_cols]) == -1).astype(int)
    daily["anomaly_score"] = -iso.score_samples(daily[kpi_cols])   # higher = more anomalous
else:
    daily["anomaly_flag"]  = 0
    daily["anomaly_score"] = 0.0

print(f"  Daily KPI rows: {len(daily):,} | Anomaly days flagged: {daily['anomaly_flag'].sum()}")
print("✅ Anomaly detection complete.")


ANOMALY DETECTION — DAILY KPIs
  Daily KPI rows: 38 | Anomaly days flagged: 2
✅ Anomaly detection complete.


In [25]:
# ===========================================================================
# CELL 11 — OPTIONAL: DROP-OFF PREDICTION MODEL
# ===========================================================================

print("\n" + "="*60)
print("DROP-OFF PREDICTION MODEL (Gradient Boosting)")
print("="*60)

# Encode categoricals for the model
from sklearn.preprocessing import LabelEncoder

ml_df = conv_agg[[
    "avg_prompt_len", "sentiment_mean", "prompt_category",
    "hour_of_day", "language", "model", "drop_off_flag"
]].copy().fillna(0)

for col in ["prompt_category", "language", "model"]:
    le = LabelEncoder()
    ml_df[col] = le.fit_transform(ml_df[col].astype(str))

X = ml_df.drop(columns="drop_off_flag")
y = ml_df["drop_off_flag"]

if len(y.unique()) == 2 and len(X) >= 100:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
    gb.fit(X_train, y_train)
    auc = roc_auc_score(y_test, gb.predict_proba(X_test)[:, 1])
    print(f"  GradientBoosting AUC on test set: {auc:.4f}")

    # Score all rows
    conv_agg["drop_off_probability"] = gb.predict_proba(X)[:, 1]
else:
    print("  Not enough data for model. Skipping.")
    conv_agg["drop_off_probability"] = np.nan

print("✅ Drop-off model complete.")



DROP-OFF PREDICTION MODEL (Gradient Boosting)
  GradientBoosting AUC on test set: 0.8118
✅ Drop-off model complete.


In [26]:
# ===========================================================================
# CELL 12 — STAGE 5: OUTPUT CSVs
# ===========================================================================

print("\n" + "="*60)
print("STAGE 5 — SAVING OUTPUT CSVs")
print("="*60)

out = CFG["output_dir"]

# ── conversations_clean.csv ──────────────────────────────────────────────────
conv_out_cols = [
    "conversation_id", "conv_turn_count", "user_msg_count",
    "avg_prompt_len", "avg_response_len", "response_quality_score",
    "sentiment_mean", "toxicity_max", "toxic_flag",
    "drop_off_flag", "drop_off_probability",
    "session_duration_min", "country", "language", "model",
    "hour_of_day", "day_of_week", "week_num",
    "prompt_category", "is_power_user",
    "cluster_id", "cluster_label", "pca_x", "pca_y",
    "ts_min", "ts_max",
]
conv_save = conv_agg[[c for c in conv_out_cols if c in conv_agg.columns]]
path_conv = os.path.join(out, "conversations_clean.csv")
conv_save.to_csv(path_conv, index=False)
print(f"  ✅ conversations_clean.csv  → {len(conv_save):,} rows")

# ── messages_sample.csv (stratified 10% of individual messages) ──────────────
sample_msgs = df.sample(frac=0.10, random_state=42)
msg_cols = [
    "conversation_id", "turn_id", "role", "content_clean",
    "token_count", "sentiment_score", "toxicity_score",
    "language", "country", "model", "hour_of_day", "day_of_week",
    "timestamp",
]
path_msgs = os.path.join(out, "messages_sample.csv")
sample_msgs[[c for c in msg_cols if c in sample_msgs.columns]].to_csv(
    path_msgs, index=False
)
print(f"  ✅ messages_sample.csv      → {len(sample_msgs):,} rows")

# ── daily_kpis.csv ───────────────────────────────────────────────────────────
path_daily = os.path.join(out, "daily_kpis.csv")
daily.to_csv(path_daily, index=False)
print(f"  ✅ daily_kpis.csv           → {len(daily):,} rows")

# ── geo_summary.csv ──────────────────────────────────────────────────────────
geo = (
    conv_agg.groupby("country")
    .agg(
        conversation_count = ("conversation_id",       "count"),
        toxicity_rate      = ("toxic_flag",            lambda x: round(x.mean() * 100, 2)),
        avg_quality_score  = ("response_quality_score","mean"),
        drop_off_rate      = ("drop_off_flag",         lambda x: round(x.mean() * 100, 2)),
        engagement_rate    = ("drop_off_flag",         lambda x: round((1-x.mean())*100, 2)),
    )
    .reset_index()
    .sort_values("conversation_count", ascending=False)
)
path_geo = os.path.join(out, "geo_summary.csv")
geo.to_csv(path_geo, index=False)
print(f"  ✅ geo_summary.csv          → {len(geo):,} rows")

# ── prompt_categories.csv ────────────────────────────────────────────────────
pcat = (
    conv_agg.groupby(["week_num", "country", "model", "prompt_category"])
    .agg(
        conversation_count  = ("conversation_id", "count"),
        avg_quality_score   = ("response_quality_score", "mean"),
        drop_off_rate       = ("drop_off_flag", lambda x: round(x.mean() * 100, 2)),
    )
    .reset_index()
)
path_pcat = os.path.join(out, "prompt_categories.csv")
pcat.to_csv(path_pcat, index=False)
print(f"  ✅ prompt_categories.csv    → {len(pcat):,} rows")

# ── user_segments.csv ────────────────────────────────────────────────────────
seg_out = conv_agg[[
    "conversation_id", "cluster_id", "cluster_label",
    "pca_x", "pca_y",
    "conv_turn_count", "avg_prompt_len", "sentiment_mean",
    "toxicity_max", "session_duration_min", "response_quality_score",
]].copy()
path_seg = os.path.join(out, "user_segments.csv")
seg_out.to_csv(path_seg, index=False)
print(f"  ✅ user_segments.csv        → {len(seg_out):,} rows")

print("\n🎉 ALL OUTPUT CSVs SAVED SUCCESSFULLY!")



STAGE 5 — SAVING OUTPUT CSVs
  ✅ conversations_clean.csv  → 99,110 rows
  ✅ messages_sample.csv      → 55,638 rows
  ✅ daily_kpis.csv           → 38 rows
  ✅ geo_summary.csv          → 159 rows
  ✅ prompt_categories.csv    → 2,315 rows
  ✅ user_segments.csv        → 99,110 rows

🎉 ALL OUTPUT CSVs SAVED SUCCESSFULLY!


In [27]:
# ===========================================================================
# CELL 13 — DOWNLOAD ALL CSVs  (Colab only)
# ===========================================================================

print("\n" + "="*60)
print("DOWNLOADING FILES TO LOCAL MACHINE")
print("="*60)

try:
    from google.colab import files

    csv_files = [
        path_conv, path_msgs, path_daily,
        path_geo, path_pcat, path_seg,
    ]

    for fpath in csv_files:
        fname = os.path.basename(fpath)
        size  = os.path.getsize(fpath) / 1024
        print(f"  ⬇️  Downloading {fname}  ({size:.1f} KB)...")
        files.download(fpath)

    print("\n✅ All files downloaded.")

except ImportError:
    print("  (Not running in Colab — files saved to:", out, ")")


DOWNLOADING FILES TO LOCAL MACHINE
  ⬇️  Downloading conversations_clean.csv  (25638.5 KB)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ⬇️  Downloading messages_sample.csv  (53811.3 KB)...


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  ⬇️  Downloading user_segments.csv  (13166.0 KB)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All files downloaded.


In [28]:
# ===========================================================================
# CELL 14 — QUICK SUMMARY REPORT
# ===========================================================================

print("\n" + "="*60)
print("PIPELINE SUMMARY")
print("="*60)

print(f"""
  Raw rows loaded           : {len(df_raw):,}
  Rows after cleaning       : {len(df):,}
  Unique conversations      : {conv_agg['conversation_id'].nunique():,}

  Drop-off rate (≤2 turns)  : {conv_agg['drop_off_flag'].mean()*100:.1f}%
  Mean turn count           : {conv_agg['conv_turn_count'].mean():.2f}
  Mean quality score        : {conv_agg['response_quality_score'].mean():.2f} / 10
  Mean toxicity rate        : {conv_agg['toxic_flag'].mean()*100:.2f}%
  Power users               : {conv_agg['is_power_user'].sum():,}
    ({conv_agg['is_power_user'].mean()*100:.1f}% of conversations)

  Anomaly days detected     : {daily['anomaly_flag'].sum()}

  Output files:
    • conversations_clean.csv
    • messages_sample.csv
    • daily_kpis.csv
    • geo_summary.csv
    • prompt_categories.csv
    • user_segments.csv
""")



PIPELINE SUMMARY

  Raw rows loaded           : 100,000
  Rows after cleaning       : 556,384
  Unique conversations      : 99,110
 
  Drop-off rate (≤2 turns)  : 49.0%
  Mean turn count           : 5.61
  Mean quality score        : 4.42 / 10
  Mean toxicity rate        : 0.00%
  Power users               : 7,880
    (8.0% of conversations)
 
  Anomaly days detected     : 2
 
  Output files:
    • conversations_clean.csv
    • messages_sample.csv
    • daily_kpis.csv
    • geo_summary.csv
    • prompt_categories.csv
    • user_segments.csv



In [29]:
# =============================================================================
# WildChat Analytics — Combined CSV Builder
# Run AFTER WildChat_ETL_Pipeline.py has completed
# Merges all 6 output CSVs into one single downloadable file
# =============================================================================

import os
import pandas as pd

OUT_DIR = "/content/wildchat_outputs"

# ── Load all 6 CSVs ───────────────────────────────────────────────────────────
print("Loading CSVs...")

conversations  = pd.read_csv(os.path.join(OUT_DIR, "conversations_clean.csv"))
user_segments  = pd.read_csv(os.path.join(OUT_DIR, "user_segments.csv"))
prompt_cats    = pd.read_csv(os.path.join(OUT_DIR, "prompt_categories.csv"))
geo_summary    = pd.read_csv(os.path.join(OUT_DIR, "geo_summary.csv"))
daily_kpis     = pd.read_csv(os.path.join(OUT_DIR, "daily_kpis.csv"))
messages       = pd.read_csv(os.path.join(OUT_DIR, "messages_sample.csv"))

print(f"  conversations_clean : {len(conversations):,} rows")
print(f"  user_segments       : {len(user_segments):,} rows")
print(f"  messages_sample     : {len(messages):,} rows")

# ── Step 1: Merge conversation + user_segments on conversation_id ─────────────
# user_segments has cluster info + PCA — already in conversations_clean,
# but we re-merge cleanly to avoid duplicate columns

seg_cols = ["conversation_id", "cluster_id", "cluster_label", "pca_x", "pca_y"]
combined = conversations.merge(
    user_segments[seg_cols],
    on="conversation_id",
    how="left",
    suffixes=("", "_seg"),
)

# Drop duplicate cluster columns if they already existed in conversations_clean
dup_cols = [c for c in combined.columns if c.endswith("_seg")]
combined.drop(columns=dup_cols, inplace=True)

# ── Step 2: Attach geo-level KPIs (country-level rollup) ─────────────────────
geo_cols = ["country", "toxicity_rate", "avg_quality_score",
            "drop_off_rate", "engagement_rate"]
combined = combined.merge(
    geo_summary[geo_cols].rename(columns={
        "toxicity_rate"    : "geo_toxicity_rate",
        "avg_quality_score": "geo_avg_quality",
        "drop_off_rate"    : "geo_drop_off_rate",
        "engagement_rate"  : "geo_engagement_rate",
    }),
    on="country",
    how="left",
)

# ── Step 3: Attach daily KPI context (join on date from ts_min) ───────────────
daily_kpis["date"] = pd.to_datetime(daily_kpis["date"]).dt.date
combined["ts_min"] = pd.to_datetime(combined["ts_min"], utc=True, errors="coerce")
combined["date"]   = combined["ts_min"].dt.date

daily_cols = ["date", "anomaly_flag", "anomaly_score",
              "total_conversations", "engagement_rate"]
combined = combined.merge(
    daily_kpis[daily_cols].rename(columns={
        "total_conversations": "day_total_conversations",
        "engagement_rate"    : "day_engagement_rate",
    }),
    on="date",
    how="left",
)

# ── Step 4: Attach first-turn message features from messages_sample ───────────
# Pull turn_id == 1, role == user  →  first prompt's token count & sentiment
first_turns = (
    messages[
        (messages["role"] == "user") &
        (messages["turn_id"] == 1)
    ][["conversation_id", "token_count", "sentiment_score"]]
    .rename(columns={
        "token_count"     : "first_prompt_tokens",
        "sentiment_score" : "first_prompt_sentiment",
    })
    .drop_duplicates("conversation_id")
)
combined = combined.merge(first_turns, on="conversation_id", how="left")

# ── Step 5: Clean up helper columns ──────────────────────────────────────────
combined.drop(columns=["date"], inplace=True, errors="ignore")

# ── Step 6: Save ─────────────────────────────────────────────────────────────
out_path = os.path.join(OUT_DIR, "wildchat_combined.csv")
combined.to_csv(out_path, index=False)

size_mb = os.path.getsize(out_path) / (1024 * 1024)
print(f"\n✅ wildchat_combined.csv saved")
print(f"   Rows    : {len(combined):,}")
print(f"   Columns : {len(combined.columns)}")
print(f"   Size    : {size_mb:.2f} MB")
print(f"\n   Columns included:\n   {list(combined.columns)}")

# ── Step 7: Download ──────────────────────────────────────────────────────────
try:
    from google.colab import files
    print("\n⬇️  Downloading wildchat_combined.csv ...")
    files.download(out_path)
    print("✅ Download triggered.")
except ImportError:
    print(f"\n(Not in Colab — file saved at: {out_path})")

Loading CSVs...
  conversations_clean : 99,110 rows
  user_segments       : 99,110 rows
  messages_sample     : 55,638 rows

✅ wildchat_combined.csv saved
   Rows    : 99,110
   Columns : 36
   Size    : 31.19 MB

   Columns included:
   ['conversation_id', 'conv_turn_count', 'user_msg_count', 'avg_prompt_len', 'avg_response_len', 'response_quality_score', 'sentiment_mean', 'toxicity_max', 'toxic_flag', 'drop_off_flag', 'drop_off_probability', 'session_duration_min', 'country', 'language', 'model', 'hour_of_day', 'day_of_week', 'week_num', 'prompt_category', 'is_power_user', 'cluster_id', 'cluster_label', 'pca_x', 'pca_y', 'ts_min', 'ts_max', 'geo_toxicity_rate', 'geo_avg_quality', 'geo_drop_off_rate', 'geo_engagement_rate', 'anomaly_flag', 'anomaly_score', 'day_total_conversations', 'day_engagement_rate', 'first_prompt_tokens', 'first_prompt_sentiment']

⬇️  Downloading wildchat_combined.csv ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download triggered.
